<a href="https://colab.research.google.com/github/saeyeon055-hue/Bio-AI-Learning-Path/blob/main/ML/4%EA%B0%95_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SMSSpamCollection 데이터를 입력으로 하는 SVM 기반 스팸 메일 필터링 프로그램 작성

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## 데이터 읽기 (처음부터 100개)

In [ ]:
import numpy as np

file_path = '/content/drive/MyDrive/Colab Notebooks/기계학습/SMSSpamCollection.txt'

In [ ]:
# 파일 읽기
x_data, y_data= [], []
with open(file_path, 'r', encoding='utf8') as inFile:
  lines=inFile.readlines()

lines = lines[:100]

for line in lines:
  line = line.strip().split('\t')
  sentence, label = line[1], line[0]
  x_data.append(sentence)
  y_data.append(label)

print('x_data의 개수 : ' + str(len(x_data)))
print('y_data의 개수 : ' + str(len(y_data)))

x_data의 개수 : 100
y_data의 개수 : 100


## 데이터 변환 (문자열 -> 숫자)

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer=Tokenizer()

# spam, ham 라벨을 대응하는 index로 치환하기위한 딕셔너리
label2index_dict={'spam':0, 'ham':1}

# indexing 한 데이터를 넣을 리스트 선언
indexing_x_data, indexing_y_data=[], []

for label in y_data:
  indexing_y_data.append(label2index_dict[label])

# x_data를 사용하여 딕셔너리 생성
tokenizer.fit_on_texts(x_data)

# x_data에 있는 각 문장의 단어들을 대응하는 index로 치환하고 그 결과값을 indexing_x_data에 저장
indexing_x_data=tokenizer.texts_to_sequences(x_data)

print('x_data indexing 하기 전 : ' + str(x_data[0]))
print('x_data indexing 하기 후 : ' + str(indexing_x_data[0]))
print('y_data indexing 하기 전 : ' + str(y_data[0]))
print('y_data indexing 하기 후 : ' + str(indexing_y_data[0]))

x_data indexing 하기 전 : Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
x_data indexing 하기 후 : [38, 93, 239, 240, 241, 242, 53, 11, 243, 72, 94, 244, 245, 126, 246, 247, 73, 74, 248, 127]
y_data indexing 하기 전 : ham
y_data indexing 하기 후 : 1


## SVM 학습

In [ ]:
from sklearn.svm import SVC
# 문장의 길이를 max_length으로 맞춰 변환
max_length = 60
for index in range(len(indexing_x_data)):
  length = len(indexing_x_data[index])

  if(length >max_length):
    indexing_x_data[index] = indexing_x_Data[index][:max_length]
  elif(length < max_length):
    indexing_x_data[index] = indexing_x_data[index] + [0]*(max_length-length)

# 전체 데이터를 9:1의 비율로 나누어 학습 및 평가 데이터로 사용
number_of_train=int(len(indexing_x_data)*0.9)

train_x=indexing_x_data[:number_of_train]
train_y=indexing_y_data[:number_of_train]
test_x=indexing_x_data[number_of_train:]
test_y=indexing_y_data[number_of_train:]

print('train_x의 개수 : ' + str(len(train_x)))
print('train_y의 개수 : ' + str(len(train_y)))
print('test_x의 개수 : ' + str(len(test_x)))
print('test_y의 개수 : ' + str(len(test_y)))

svm=SVC(kernel='linear', C=1e10)
svm.fit(train_x,train_y)

train_x의 개수 : 90
train_y의 개수 : 90
test_x의 개수 : 10
test_y의 개수 : 10


SVC(C=10000000000.0, kernel='linear')

## SVM 평가

In [ ]:
predict =svm.predict(test_x)

correct_count=0
for index in range(len(predict)):
  if(test_y[index] == predict[index]):
    correct_count+=1

accuracy = 100.0*correct_count/len(test_y)

print('Accuracy: ' + str(accuracy))

index2label = {0:'spam', 1:'ham'}

test_x_word = tokenizer.sequences_to_texts(test_x)

for index in range(len(test_x_word)):
  print()
  print('문장 : ', test_x_word[index])
  print('정답 : ', index2label[test_y[index]])
  print('모델 출력 : ', index2label[predict[index]])

Accuracy: 80.0

문장 :  yeah do don‘t stand to close tho you‘ll catch something
정답 :  ham
모델 출력 :  spam

문장 :  sorry to be a pain is it ok if we meet another night i spent late afternoon in casualty and that means i haven't done any of y stuff42moro and that includes all my time sheets and that sorry
정답 :  ham
모델 출력 :  spam

문장 :  smile in pleasure smile in pain smile when trouble pours like rain smile when sum1 hurts u smile becoz someone still loves to see u smiling
정답 :  ham
모델 출력 :  ham

문장 :  please call our customer service representative on 0800 169 6031 between 10am 9pm as you have won a guaranteed ￡1000 cash or ￡5000 prize
정답 :  spam
모델 출력 :  spam

문장 :  havent planning to buy later i check already lido only got 530 show in e afternoon u finish work already
정답 :  ham
모델 출력 :  ham

문장 :  your free ringtone is waiting to be collected simply text the password mix to 85069 to verify get usher and britney fml po box 5249 mk17 92h 450ppw 16
정답 :  spam
모델 출력 :  spam

문장 :  watching tel